In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import torch
import evaluate

df = pd.read_csv('../data/cleaned_pushshift.csv')
print(f"Total samples: {len(df)}")
print(df['dialect'].value_counts())

Total samples: 2748382
dialect
rioplatense    1227656
cdmx           1214181
madrid          306545
Name: count, dtype: int64


REDUCING SIZE DUE TO LARGE PROCESSING TIMES, UNFORTUNATELY AFTER REPEATED CRASHES WITH MY SYSTEM AND COLAB RUNNING OUT OF TIME I REDUCED THE SIZE SIGNIFICANTLY, IF INTERESTED THE FULL PROCESSED DATA IS AVAILABLE AND YOU CAN JUST SKIP THE NEXT SNIPPET (ALSO DIALECT ID IS NOT THE MAIN CONCERN OF THIS STUDY)

In [ ]:
targets = {
    'madrid': 50000,
    'rioplatense': 200000,
    'cdmx': 200000
} 

samples = []

for dialect, count in targets.items():
    subset = df[df['dialect'] == dialect]
    
    # Check if we have enough rows; if not, take everything available
    n_to_sample = min(len(subset), count)
    
    if n_to_sample < count:
        print(f"Warning: Only found {len(subset)} rows for {dialect}. Taking all of them.")
    
    sampled_subset = subset.sample(n=n_to_sample, random_state=42)
    samples.append(sampled_subset)

df_dev = pd.concat(samples).reset_index(drop=True)

print(f"New dataset size: {len(df_dev)}")
print(df_dev['dialect'].value_counts())

New dataset size: 450000
dialect
rioplatense    200000
cdmx           200000
madrid          50000
Name: count, dtype: int64


In [ ]:
dialect_to_id = {'rioplatense': 0, 'cdmx': 1, 'madrid': 2}
df_dev['labels'] = df_dev['dialect'].map(dialect_to_id)

train_df, val_df = train_test_split(
    df_dev, test_size=0.2, stratify=df_dev['dialect'], random_state=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print("Train dialect distribution:")
print(train_df['dialect'].value_counts())

Train: 360000, Val: 90000
Train dialect distribution:
dialect
rioplatense    160000
cdmx           160000
madrid          40000
Name: count, dtype: int64


In [ ]:
MODEL_CHECKPOINT = "dccuchile/bert-base-spanish-wwm-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    return tokenizer(
        examples['text'], 
        truncation=True, 
        padding='max_length', 
        max_length=MAX_LENGTH
    )

train_dataset = Dataset.from_pandas(train_df[['text', 'labels']])
val_dataset = Dataset.from_pandas(val_df[['text', 'labels']])

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/360000 [00:00<?, ? examples/s]

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, 
    num_labels=3
)

accuracy = evaluate.load("accuracy")
id2label = {0: 'rioplatense', 1: 'cdmx', 2: 'madrid'}
label2id = {'rioplatense': 0, 'cdmx': 1, 'madrid': 2}
model.config.id2label = id2label
model.config.label2id = label2id

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

In [35]:
training_args = TrainingArguments(
    output_dir='./dialect_final_fast',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=400,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=0,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

trainer.save_model("./dialect_classifier_final")
tokenizer.save_pretrained("./dialect_classifier_final")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.382128,0.400015,0.834022


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./dialect_classifier_final\\tokenizer_config.json',
 './dialect_classifier_final\\tokenizer.json')

In [ ]:
from transformers import pipeline
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

classifier = pipeline(
    "text-classification",
    model="./dialect_classifier_final",
    tokenizer="./dialect_classifier_final",
    return_all_scores=True
)

test_texts = [
    "Che, este link está buenísimo boludo",
    "¡Qué chido carnal, ya vi el video!",
    "Joder tío, qué gracia me ha hecho esto"
]

print("Testing classifier...")
for text in test_texts:
    result = classifier(text)
    top_pred = result[0] if isinstance(result[0], dict) else max(result, key=lambda x: x['score'])
    print(f"\nText: '{text}'")
    print(f"Predicted: {top_pred['label']} (confidence: {top_pred['score']:.1%})")

print("Testing 100 comments...")
sample_texts = train_df['text'].sample(100).tolist()
batch_results = classifier(
    sample_texts, 
    truncation=True, 
    max_length=256, 
    batch_size=8
)

rio, cdmx, madrid = 0, 0, 0
for result in batch_results:

    if isinstance(result, dict):
        top_pred = result  # Direct dict
    elif isinstance(result, list):
        if len(result) > 0:
            first_item = result[0]
            if isinstance(first_item, dict):
                top_pred = first_item  # List of dicts [{}]
            else:
                top_pred = max(first_item, key=lambda x: x['score'])  # List of lists [[{},{},{}]]
        else:
            continue  # Skip empty
    else:
        continue  # Skip invalid
    
    label = top_pred['label'].lower()
    if 'rioplatense' in label: rio += 1
    elif 'cdmx' in label: cdmx += 1
    elif 'madrid' in label: madrid += 1

print(f"Results: Rio:{rio}% | CDMX:{cdmx}% | Madrid:{madrid}% | Total:{(rio+cdmx+madrid)/100:.0%}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Dialect classifier ready! Testing...

Text: 'Che, este link está buenísimo boludo'
Predicted: rioplatense (confidence: 99.6%)

Text: '¡Qué chido carnal, ya vi el video!'
Predicted: cdmx (confidence: 99.8%)

Text: 'Joder tío, qué gracia me ha hecho esto'
Predicted: madrid (confidence: 94.8%)

🔍 Batch testing 100 comments...
Results: Rio:44% | CDMX:50% | Madrid:6% | Total:100%
